In [ ]:
import pandas as pd
df = pd.read_csv("6-classification-data/SMSSpamCollection.tsv", sep='\t', header=None, names=["Label", "Text"])
print(df["Label"].value_counts())

def create_balanced_dataset(df):
    
    # Count the instances of "spam"
    num_spam = df[df["Label"] == "spam"].shape[0]
    
    # Randomly sample "ham" instances to match the number of "spam" instances
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    
    # Combine ham "subset" with "spam"
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df


balanced_df = create_balanced_dataset(df)
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})    

print(balanced_df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64
Label
0    747
1    747
Name: count, dtype: int64


In [2]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [3]:
def random_split(df, train_frac, validation_frac):
    # Shuffle the entire DataFrame
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # Calculate split indices
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # Split the DataFrame
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
# Test size is implied to be 0.2 as the remainder

train_df.to_csv("6-classification-data/train.csv", index=None)
validation_df.to_csv("6-classification-data/validation.csv", index=None)
test_df.to_csv("6-classification-data/test.csv", index=None)

In [4]:
import torch
from torch.utils.data import Dataset


class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # Pre-tokenize texts
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # Truncate sequences if they are longer than max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        return max(len(encoded_text) for encoded_text in self.encoded_texts)
    
train_dataset = SpamDataset(
    csv_file="6-classification-data/train.csv",
    max_length=None,
    tokenizer=tokenizer
)

val_dataset = SpamDataset(
    csv_file="6-classification-data/validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="6-classification-data/test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)



In [5]:
for i, t in train_loader:
    print(i.shape)
    print(t.shape)
    break

torch.Size([8, 120])
torch.Size([8])


In [6]:
print("Train loader:")
for input_batch, target_batch in train_loader:
    pass

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

Train loader:
Input batch dimensions: torch.Size([8, 120])
Label batch dimensions torch.Size([8])
130 training batches
19 validation batches
38 test batches


In [7]:
from llms_from_scratch_utils import GPT_124M_FILE, GPT_MODEL_CONFIGS, BASE_GPT_CONFIG
from llms_from_scratch_core import GPTModel
BASE_GPT_CONFIG.update(GPT_MODEL_CONFIGS["gpt2-small (124M)"])

gpt = GPTModel(BASE_GPT_CONFIG)
gpt.load_state_dict(torch.load(GPT_124M_FILE, weights_only=True))
gpt.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, 

## Classification Head

In [8]:
# freeze all layers
for param in gpt.parameters():
    param.requires_grad = False

torch.manual_seed(123)

# change the output head layer to be 768 x 2
num_classes = 2
gpt.out_head = torch.nn.Linear(in_features=BASE_GPT_CONFIG["emb_dim"], out_features=num_classes)

# make the last transformer block and layernom trainable
for param in gpt.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in gpt.final_norm.parameters():
    param.requires_grad = True

In [9]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)

with torch.no_grad():
    outputs = gpt(inputs)

outputs

seq_len 4
x.shape torch.Size([1, 4, 768])
mha context_vec torch.Size([1, 4, 768])
mha out_proj_result torch.Size([1, 4, 768])
ff x.shape torch.Size([1, 4, 768])
Linear(in_features=768, out_features=3072, bias=True)
mha context_vec torch.Size([1, 4, 768])
mha out_proj_result torch.Size([1, 4, 768])
ff x.shape torch.Size([1, 4, 768])
Linear(in_features=768, out_features=3072, bias=True)
mha context_vec torch.Size([1, 4, 768])
mha out_proj_result torch.Size([1, 4, 768])
ff x.shape torch.Size([1, 4, 768])
Linear(in_features=768, out_features=3072, bias=True)
mha context_vec torch.Size([1, 4, 768])
mha out_proj_result torch.Size([1, 4, 768])
ff x.shape torch.Size([1, 4, 768])
Linear(in_features=768, out_features=3072, bias=True)
mha context_vec torch.Size([1, 4, 768])
mha out_proj_result torch.Size([1, 4, 768])
ff x.shape torch.Size([1, 4, 768])
Linear(in_features=768, out_features=3072, bias=True)
mha context_vec torch.Size([1, 4, 768])
mha out_proj_result torch.Size([1, 4, 768])
ff x.shap

tensor([[[-1.5854,  0.9904],
         [-3.7235,  7.4548],
         [-2.2661,  6.6049],
         [-3.5983,  3.9902]]])

## Training Loop

In [10]:
def calc_loss_batch(input_batch, target_batch, model):
    logits = model(input_batch)[:,-1,:] # logits of last output token
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

def calc_accuracy_loader(data_loader, model, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            with torch.no_grad():
                logits = model(input_batch)[:,-1,:] # logits of last output token in sequence
            predicted_labels = torch.argmax(logits, dim=-1)
            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

def train_classifier_simple(model, train_loader, val_loader, optimizer, 
                            num_epochs):
    examples_seen, global_step = 0,-1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1
        train_accuracy = calc_accuracy_loader(train_loader, model)
        val_accuracy = calc_accuracy_loader(val_loader, model)
        print(f"Training accuracy: {train_accuracy}")
        print(f"Val accuracy {val_accuracy}")
        return model

import time
start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(gpt.parameters(), lr=5e-5, weight_decay=0.1)
train_classifier_simple(gpt, train_loader, val_loader, optimizer, num_epochs=1)
print(f"Training completed in {time.time() - start_time}")

In [ ]:
from llms_from_scratch_utils import generate, generate_and_print_sample
import tiktoken
print(generate(gpt, "Every effort moves you", max_new_tokens=15, context_size=BASE_CONFIG["context_length"], top_k=25, temperature=1.4))

tokenizer = tiktoken.get_encoding("gpt2")
generate_and_print_sample(model=gpt, start_context="Every effort moves you", tokenizer=tokenizer)


Every effort moves you up to this goal and your efforts lead them down into the abyss. So
Every effort moves you forward.  The first step is to understand the importance of your work.  The second step is to understand the importance of your work.  The third step is to understand the importance of your work.  The fourth step is


In [ ]:
def classify_review(text, model, tokenizer, pad_token_id=50256):
    model.eval()
    input = tokenizer.encode(text)
    supported_ctx_len = model.pos_emb.weight.shape[0]
    print(supported_ctx_len)
    input = input[:supported_ctx_len]
    input += [pad_token_id]*(120 - len(input))
    input_tensor = torch.tensor(input).unsqueeze(0)
    with torch.no_grad():
        logits = model(input_tensor)[:,-1,:]
    predicted = torch.argmax(logits, dim=-1).item()
    return "spam" if predicted == 1 else "ham"

In [ ]:

text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

print(classify_review(text_1, gpt, tokenizer))


text_2 = (
    "Please send me your bank details to transfer the reward"
)

print(classify_review(val_dataset.data["Text"][0], gpt, tokenizer))
print(classify_review(val_dataset.data["Text"][1], gpt, tokenizer))
print(classify_review(val_dataset.data["Text"][2], gpt, tokenizer))

1024
spam
1024
spam
1024
spam
1024
ham


In [ ]:
val_dataset.data


,Label,Text
0,1,"Mila, age23, blonde, new in UK. I look sex wit..."
1,1,"Hungry gay guys feeling hungry and up 4 it, no..."
2,0,Ugh. Gotta drive back to sd from la. My butt i...
3,0,Please leave this topic..sorry for telling that..
4,1,We tried to contact you re our offer of New Vi...
...,...,...
144,0,Hey gorgeous man. My work mobile number is. Ha...
145,0,Just sleeping..and surfing
146,0,"I'm in solihull, | do you want anything?"
147,0,"Jay told me already, will do"


In [ ]:
a = [1,2,3,4]

a[:1000]

[1, 2, 3, 4]